In [0]:
%run "../includes/common_functions"

In [0]:
%sql
USE e_commerce.bronze;

In [0]:
customer_df = spark.read.format("delta").table("bronze.customer")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.customer(
  customer_id VARCHAR(50),
  zip_code_prefix VARCHAR(40),
  city VARCHAR(40),
  state VARCHAR(40),
  ingestion_timestamp TIMESTAMP,
  environment VARCHAR(50),
  file_date DATE,
  source_name STRING
)
USING DELTA
LOCATION 'abfss://silver@saexternaldatabricks.dfs.core.windows.net/customer'

In [0]:
reglas = []

In [0]:
from pyspark.sql.functions import col, current_timestamp
import json

In [0]:
nulos(customer_df, "customer_id", reglas, "customer")
duplicados(customer_df, "customer_id", reglas, "customer")

In [0]:
df_reglas = spark.createDataFrame(reglas)
df_reglas = df_reglas.withColumn("validation_date", current_timestamp())

In [0]:
df_reglas.write.mode("append").format("delta").saveAsTable("bronze.log_transformation")

In [0]:
error = [regla for regla in reglas if not regla["cumple"]]

if error:
    dbutils.jobs.taskValues.set(key="estado", value="Error crítico")
    dbutils.jobs.taskValues.set(key="detalle", value=json.dumps(error))
else:
    dbutils.jobs.taskValues.set(key="estado", value="Validación exitosa")

In [0]:
%sql
SELECT * FROM bronze.log_transformation;

In [0]:
last_record = spark.sql("""
SELECT COALESCE(MAX(ingestion_timestamp), '1900-01-01')
FROM e_commerce.silver.customer
""").collect()[0][0]

df_incremental = spark.read.table("e_commerce.bronze.customer") \
    .filter(f"ingestion_timestamp > '{last_record}'")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = Window.partitionBy("customer_id") \
               .orderBy(col("ingestion_timestamp").desc())

df_dedup = df_incremental \
    .withColumn("rn", row_number().over(window)) \
    .filter("rn = 1") \
    .drop("rn")

In [0]:
df_dedup.createOrReplaceTempView("updates")

In [0]:
if df_dedup.count() > 0:
    spark.sql("""
        MERGE INTO e_commerce.silver.customer t
        USING updates s
        ON t.customer_id = s.customer_id

        WHEN MATCHED AND s.ingestion_timestamp > t.ingestion_timestamp THEN
        UPDATE SET
            t.zip_code_prefix = s.zip_code_prefix,
            t.city = INITCAP(s.city),
            t.state = s.state,
            t.ingestion_timestamp = s.ingestion_timestamp,
            t.environment = s.environment,
            t.file_date = s.file_date

        WHEN NOT MATCHED THEN
        INSERT (
            customer_id,
            zip_code_prefix,
            city,
            state,
            ingestion_timestamp,
            environment,
            file_date,
            source_name
        )
        VALUES (
            s.customer_id,
            s.zip_code_prefix,
            INITCAP(s.city),
            s.state,
            s.ingestion_timestamp,
            s.environment,
            s.file_date,
            s.source_name
        );
    """)

In [0]:
%sql
SELECT file_date, COUNT(1) FROM silver.customer GROUP BY file_date;